In [ ]:
#Clone the project repo (skip if running locally from the repo root)
#Note: the training CSVs are not stored in the repo - regenerate arxiv_abstracts_cleaned.csv
#by running 02_scraper.ipynb and 03_cleaning_eda.ipynb first, then place it in backend/data/
!git clone https://github.com/expaetra/CM_3070_NLP.git
%cd CM_3070_NLP

A range of classifiers is evaluated to compare their performance on text classification.:
- linear models (Logistic Regression and Linear SVM),
- probabilistic model (Complement Naive Bayes)
- tree-based model (Random Forest)

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB

import nltk
from nltk.corpus import stopwords

In [ ]:
# load cleaned dataset
df = pd.read_csv("backend/data/arxiv_abstracts_cleaned.csv")
print(df.shape)
print(df.head())

(15717, 5)
  category             field               discipline  \
0    cs.LG  Machine Learning  Artificial Intelligence   
1    cs.LG  Machine Learning  Artificial Intelligence   
2    cs.LG  Machine Learning  Artificial Intelligence   
3    cs.LG  Machine Learning  Artificial Intelligence   
4    cs.LG  Machine Learning  Artificial Intelligence   

                                            abstract  abstract_length  
0  design rule checking (drc) is getting increasi...              117  
1  anomaly detection is a key application of mach...              125  
2  the increased reliance on the internet and the...              220  
3  clinical coding is an administrative process t...              159  
4  the translation of medical diagnosis to clinic...              154  


TF-IDF word unigrams achieved the strongest overall results in the representation experiments, so this representation will be used.

In [ ]:
nltk.download("stopwords")

custom_stops = {
    'based', 'paper', 'show', 'results', 'problem', 'using', 'approach',
    'proposed', 'method', 'methods', 'propose', 'present', 'work',
    'used', 'use', 'two', 'one', 'new', 'also', 'shows', 'however',
    'provide', 'study', 'task', 'tasks', 'different', 'high', 'given'
}
stop_words = set(stopwords.words('english')).union(custom_stops)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# prepare text and labels
X_text = df["abstract"].astype(str)
y_disc = df["discipline"]
y_field = df["field"]

In [ ]:
# train/test split
X_train_text, X_test_text, y_disc_train, y_disc_test, y_field_train, y_field_test = train_test_split(
    X_text,
    y_disc,
    y_field,
    test_size=0.2,
    random_state=42,
    stratify=y_disc
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))

Train size: 12573
Test size: 3144


In [ ]:
# best representation: TF-IDF word unigrams
vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1,2),
    max_features=15000
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (12573, 15000)
Test shape: (3144, 15000)


In [ ]:
# classifiers
models = {
    'Logistic Regression': LogisticRegression(
    max_iter=3000,
    random_state=42,
    class_weight='balanced',
    solver='liblinear'
),
    'Linear SVM': LinearSVC(C=1.0, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Complement NB': ComplementNB()
}

In [ ]:
# train model and evaluate its performance

def evaluate_model(model, X_train, X_test, y_train, y_test, task_name, model_name):

    # train model
    model.fit(X_train, y_train)

    #predict on test data
    y_pred = model.predict(X_test)

    # calculate macro-averaged precision, recall, F1
    accuracy = accuracy_score(y_test,y_pred)
    macro_p, macro_r, macro_f1, _ =precision_recall_fscore_support(
        y_test, y_pred, average='macro', zero_division=0
    )
    # calculate weighted averaged precision, recall, F1
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted',zero_division=0
    )

    return {
        "Task": task_name,
        "Model": model_name,
        "Representation": "TF-IDF word unigrams",
        "Accuracy": round(accuracy, 3),
        "Macro Precision": round(macro_p, 3),
        "Macro Recall": round(macro_r, 3),
        "Macro F1": round(macro_f1, 3),
        "Weighted Precision": round(weighted_p, 3),
        "Weighted Recall": round(weighted_r, 3),
        "Weighted F1": round(weighted_f1, 3)
    }

In [ ]:
# run classifiers

results = []

for model_name, model in models.items():
    print(f"Running {model_name} - discipline")
    results.append(
        evaluate_model(
            model, X_train, X_test,
            y_disc_train, y_disc_test,
            "Discipline",model_name
        )
    )

for model_name, model in models.items():
    print(f"Running {model_name} - field")
    results.append(
        evaluate_model(
            model, X_train, X_test,
            y_field_train,y_field_test,
            "Field", model_name
        )
    )


print("All classifiers completed.")

Running Logistic Regression - discipline
Running Linear SVM - discipline
Running Random Forest - discipline


In [ ]:
# results table

results_df =pd.DataFrame(results)
results_df = results_df.sort_values(["Task", "Macro F1"], ascending=[True, False])
print(results_df)
results_df

In [ ]:
# save
os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)

results_df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/classifier_results.csv",
    index=False
)

print("Saved")